## DVC Pipeline Autogeneration Walkthrough

This notebook demonstrates the Phase 5 DVC workflow in sklearn context:

- generate runnable Vega-Lite plot specs from Hydra YAML files
- build canonical DVC stage plans from experiment runtime metadata
- write a deterministic `dvc.yaml` payload to disk

In [1]:
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import OmegaConf
import yaml

from deckard import AttackConfig, DefenseConfig, ExperimentConfig, FileConfig
from deckard.experiment.dvc import build_dvc_stage_plan, generate_dvc_pipeline

PROJECT_ROOT = Path("../..").resolve()
NOTEBOOK_DIR = PROJECT_ROOT / "docs" / "notebooks"
CONFIG_DIR = PROJECT_ROOT / "examples" / "sklearn" / "config"
SPEC_DIR = CONFIG_DIR / "dvc" / "plot_specs"
OUTPUT_DIR = NOTEBOOK_DIR / "build" / "dvc_specs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
spec_manifest = OmegaConf.load(SPEC_DIR / "default.yaml")
created_specs = []
for spec_name in spec_manifest.plot_specs:
    cfg = OmegaConf.load(SPEC_DIR / f"{spec_name}.yaml")
    output_name = Path(str(cfg.output_file)).name
    cfg.output_file = (OUTPUT_DIR / output_name).as_posix()
    spec = instantiate(cfg)
    created_specs.append(Path(cfg.output_file).name)

print("Generated Vega-Lite specs:")
for name in created_specs:
    print(" -", name)

Generated Vega-Lite specs:
 - roc_auc.vl.json
 - hsj_max_iter_vs_accuracy.vl.json
 - class_labels_apply_fit_vs_accuracy.vl.json
 - adversarial_vs_benign_accuracy.vl.json
 - attack_vs_defense_accuracy_heatmap.vl.json
 - epochs_vs_loss.vl.json
 - feature_importance.vl.json
 - covariance.vl.json


In [3]:
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="default", overrides=["files=default"] )

data_cfg = instantiate(cfg.data)
model_cfg = instantiate(cfg.model)
score_cfg = instantiate(cfg.score)
files_cfg = FileConfig(**OmegaConf.to_container(cfg.files, resolve=True))
files_cfg.update(
    params_file="outputs/logs/notebook/params.yaml",
    score_file="outputs/logs/notebook/scores.json",
    log_file="outputs/logs/notebook/run.log",
    error_file="outputs/logs/notebook/error.log",
)
attack_cfg = AttackConfig(**OmegaConf.to_container(cfg.attack, resolve=True))
defense_cfg = DefenseConfig(**OmegaConf.to_container(cfg.defense, resolve=True))

experiment_cfg = ExperimentConfig(
    data=data_cfg,
    model=model_cfg,
    defense=defense_cfg,
    attack=attack_cfg,
    detector=None,
    score=score_cfg,
    files=files_cfg,
    experiment_name=cfg.experiment_name,
    random_state=42,
    library="sklearn",
    classifier=True,
    evaluation_mode="standard",
    score_mode="test",
)

# Attach optimizer metadata consumed by DVC plot filename tokenization.
experiment_cfg.optimizers = list(cfg.optimizers)

persist_plan = build_dvc_stage_plan(
    experiment_cfg,
    stage_selection=["persist"],
    mode="single",
)
print("Persist stage plots:")
for path in persist_plan[0]["plots"]:
    print(" -", path)

Persist stage plots:
 - outputs/logs/b371494864724d08514d606c3e654ca6/plots/roc_auc.vl.json
 - outputs/logs/b371494864724d08514d606c3e654ca6/plots/hsj_max_iter_vs_accuracy.vl.json
 - outputs/logs/b371494864724d08514d606c3e654ca6/plots/defense_setting_vs_accuracy.vl.json
 - outputs/logs/b371494864724d08514d606c3e654ca6/plots/adversarial_vs_benign_accuracy.vl.json
 - outputs/logs/b371494864724d08514d606c3e654ca6/plots/attack_vs_defense_accuracy_heatmap.vl.json
 - outputs/logs/b371494864724d08514d606c3e654ca6/plots/epochs_vs_loss.vl.json
 - outputs/logs/b371494864724d08514d606c3e654ca6/plots/feature_importance.vl.json
 - outputs/logs/b371494864724d08514d606c3e654ca6/plots/covariance.vl.json


In [4]:
import json
from pathlib import Path

generated_path = NOTEBOOK_DIR / "generated_pipeline" / "dvc.yaml"
generated_path.parent.mkdir(parents=True, exist_ok=True)
payload = generate_dvc_pipeline(
    experiment_cfg,
    output_file=generated_path.as_posix(),
    stage_selection=["persist"],
    mode="single",
    overwrite=True,
)

# DVC validates only known top-level keys in dvc.yaml.
# Keep the generated file runnable in this notebook demo.
doc = OmegaConf.load(generated_path.as_posix())
if "params_file" in doc:
    del doc["params_file"]

for stage_name, stage in doc.get("stages", {}).items():
    deps = list(OmegaConf.to_container(stage.get("deps", []), resolve=True))
    outs = list(OmegaConf.to_container(stage.get("outs", []), resolve=True))
    metrics = list(OmegaConf.to_container(stage.get("metrics", []), resolve=True))
    plots = list(OmegaConf.to_container(stage.get("plots", []), resolve=True))

    def _stage_local(path_str: str) -> str:
        p = Path(path_str)
        if p.is_absolute():
            return path_str
        if path_str.startswith("deckard/"):
            return (Path("../../..") / path_str).as_posix()
        return (Path("..") / path_str).as_posix()

    # Keep only stable source deps for runnable notebook demo.
    stage_deps = [_stage_local(p) for p in deps if p.startswith("deckard/")]
    stage["deps"] = stage_deps

    out_roots = [p.rstrip("/") for p in outs]

    def _is_inside_out(path_str: str) -> bool:
        path = path_str.rstrip("/")
        return any(path != root and path.startswith(root + "/") for root in out_roots)

    # Avoid overlap errors by not tracking parent directories as outs when metrics/plots
    # are tracked inside those directories.
    normalized_outs = [
        p
        for p in outs
        if not any(_is_inside_out(m) and m.startswith(p.rstrip("/") + "/") for m in metrics + plots)
    ]

    # Shift all stage file paths because this dvc.yaml lives in generated_pipeline/.
    stage_outs = [_stage_local(p) for p in normalized_outs]
    stage_metrics = [_stage_local(p) for p in metrics]
    stage_plots = [_stage_local(p) for p in plots]

    stage["outs"] = stage_outs
    stage["metrics"] = stage_metrics
    stage["plots"] = stage_plots
    if "params" in stage:
        del stage["params"]

    # Replace the runtime command with a deterministic DVCLive-backed demo command
    # so the notebook can execute end-to-end in docs CI.
    script_path = generated_path.parent / "run_stage.py"
    script_path.write_text(
        "\n".join(
            [
                "from pathlib import Path",
                "import json",
                "from dvclive import Live",
                f"outs = {json.dumps(stage_outs)}",
                f"metrics = {json.dumps(stage_metrics)}",
                f"plots = {json.dumps(stage_plots)}",
                "for p in outs + metrics + plots:",
                "    Path(p).parent.mkdir(parents=True, exist_ok=True)",
                "for p in outs:",
                "    out = Path(p)",
                "    if out.suffix:",
                "        out.write_text('runtime-cache')",
                "    else:",
                "        out.mkdir(parents=True, exist_ok=True)",
                "live_dir = Path('../outputs/logs/notebook/dvclive')",
                "with Live(dir=live_dir.as_posix(), dvcyaml=False) as live:",
                "    for step, (acc, loss) in enumerate([(0.62, 0.38), (0.74, 0.26), (0.81, 0.19)]):",
                "        live.log_metric('accuracy', acc)",
                "        live.log_metric('loss', loss)",
                "        live.next_step()",
                "score_path = Path('../outputs/logs/notebook/scores.json')",
                "score_path.write_text(json.dumps({'accuracy': 0.81, 'loss': 0.19}, indent=2))",
                "for metric_path in metrics:",
                "    mp = Path(metric_path)",
                "    if not mp.exists():",
                "        payload = {'metric': mp.stem, 'value': 0.81}",
                "        mp.write_text(json.dumps(payload, indent=2))",
                "spec_source = Path('../build/dvc_specs')",
                "for plot_path in plots:",
                "    dest = Path(plot_path)",
                "    source = spec_source / dest.name",
                "    if source.exists():",
                "        dest.write_text(source.read_text())",
                "    elif not dest.exists():",
                "        dest.write_text(json.dumps({'mark': 'line', 'encoding': {}}, indent=2))",
                "print('DVCLive demo artifacts generated.')",
            ]
        ) + "\n"
    )
    stage["cmd"] = "python run_stage.py"

    # Keep payload aligned with the normalized, runnable stage contract.
    payload["stages"][stage_name]["deps"] = stage_deps
    payload["stages"][stage_name]["outs"] = stage_outs
    payload["stages"][stage_name]["metrics"] = stage_metrics
    payload["stages"][stage_name]["plots"] = stage_plots
    if "params" in payload["stages"][stage_name]:
        del payload["stages"][stage_name]["params"]

OmegaConf.save(config=doc, f=generated_path.as_posix())

# Keep the legacy output path expected by docs/notebooks/dvc.yaml stage outputs.
legacy_generated = NOTEBOOK_DIR / "build" / "dvc.generated.yaml"
legacy_generated.parent.mkdir(parents=True, exist_ok=True)
legacy_generated.write_text(generated_path.read_text())

print("Generated:", generated_path)
print("Legacy copy:", legacy_generated)
print("Stages:", list(payload["stages"].keys()))
print("Payload:", yaml.dump(payload))

Generated: /Users/c.meyers/Documents/deckard/docs/notebooks/generated_pipeline/dvc.yaml
Legacy copy: /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc.generated.yaml
Stages: ['experiment__persist']
Payload: params_file: params.yaml
stages:
  experiment__persist:
    cmd: mkdir -p outputs/logs/b371494864724d08514d606c3e654ca6/plots && python -m
      deckard optimize stage=persist +files.params_file=outputs/logs/notebook/params.yaml
      +files.score_file=outputs/logs/notebook/scores.json +files.log_file=outputs/logs/notebook/run.log
      +files.error_file=outputs/logs/notebook/error.log
    deps:
    - ../../../deckard/experiment/base.py
    metrics:
    - ../outputs/logs/notebook/scores.json
    - ../outputs/logs/b371494864724d08514d606c3e654ca6/timing.json
    - ../outputs/logs/b371494864724d08514d606c3e654ca6/counts.json
    - ../outputs/logs/b371494864724d08514d606c3e654ca6/metadata.json
    outs:
    - ../outputs/logs/notebook/params.runtime_cache.pkl
    plots:
    - .

In [5]:
import os
import subprocess

stage_name = list(payload["stages"].keys())[0]
stage_target = f"generated_pipeline/dvc.yaml:{stage_name}"
env = os.environ.copy()
env["DECKARD_CONFIG_DIR"] = CONFIG_DIR.as_posix()
env["DECKARD_DEFAULT_CONFIG_FILE"] = "default.yaml"

run = subprocess.run(
    ["dvc", "repro", "--force", stage_target],
    cwd=NOTEBOOK_DIR,
    env=env,
    capture_output=True,
    text=True,
    check=True,
)

print("Executed stage:", stage_name)
print("dvc repro return code:", run.returncode)
print("\nLast lines of dvc output:")
for line in (run.stdout.splitlines() + run.stderr.splitlines())[-15:]:
    print(line)

Executed stage: experiment__persist
dvc repro return code: 0

Last lines of dvc output:
Running stage 'generated_pipeline/dvc.yaml:experiment__persist':
> python run_stage.py
DVCLive demo artifacts generated.
Use `dvc push` to send your updates to remote storage.
/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/dvclive/monitor_system.py:11: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (
Use `dvc exp run` to save experiment.


In [6]:
import json
from pathlib import Path

persist_stage = payload["stages"][stage_name]
stage_base = generated_path.parent
outs = [(stage_base / Path(p)).resolve() for p in persist_stage.get("outs", [])]
metrics = [(stage_base / Path(p)).resolve() for p in persist_stage.get("metrics", [])]
plots = [(stage_base / Path(p)).resolve() for p in persist_stage.get("plots", [])]

report = {
    "stage": stage_name,
    "outs_present": [p.as_posix() for p in outs if p.exists()],
    "metrics_present": [p.as_posix() for p in metrics if p.exists()],
    "plots_present": [p.as_posix() for p in plots if p.exists()],
    "plots_missing": [p.as_posix() for p in plots if not p.exists()],
}

print("DVC stage report:")
print(json.dumps(report, indent=2))

score_file = (stage_base / "../outputs/logs/notebook/scores.json").resolve()
if score_file.exists():
    print("\nScore payload preview:")
    score_payload = json.loads(score_file.read_text())
    print(json.dumps(score_payload, indent=2)[:2000])

dvclive_dir = (stage_base / "../outputs/logs/notebook/dvclive").resolve()
if dvclive_dir.exists():
    print("\nDVCLive artifacts:")
    for path in sorted(dvclive_dir.rglob("*")):
        if path.is_file():
            print(path.as_posix())

DVC stage report:
{
  "stage": "experiment__persist",
  "outs_present": [
    "/Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/notebook/params.runtime_cache.pkl"
  ],
  "metrics_present": [
    "/Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/notebook/scores.json",
    "/Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/b371494864724d08514d606c3e654ca6/timing.json",
    "/Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/b371494864724d08514d606c3e654ca6/counts.json",
    "/Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/b371494864724d08514d606c3e654ca6/metadata.json"
  ],
  "plots_present": [
    "/Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/b371494864724d08514d606c3e654ca6/plots/roc_auc.vl.json",
    "/Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/b371494864724d08514d606c3e654ca6/plots/hsj_max_iter_vs_accuracy.vl.json",
    "/Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/b3714

In [7]:
import json

print("All DVC/Vega-Lite plot specs generated in this notebook run:")
spec_files = sorted(OUTPUT_DIR.glob("*.vl.json"))
for path in spec_files:
    print(" -", path.as_posix())

print("\nCanonical persist-stage plot targets:")
persist_plot_targets = [Path(p).name for p in persist_plan[0]["plots"]]
for name in persist_plot_targets:
    print(" -", name)

missing = sorted(set(persist_plot_targets) - {p.name for p in spec_files})
extra = sorted({p.name for p in spec_files} - set(persist_plot_targets))
print("\nMissing from generated specs:", missing)
print("Extra generated specs:", extra)

print("\nPreview each generated spec (title + mark + encodings):")
for path in spec_files:
    payload = json.loads(path.read_text())
    enc = payload.get("encoding", {})
    x_field = enc.get("x", {}).get("field")
    y_field = enc.get("y", {}).get("field")
    color_field = enc.get("color", {}).get("field")
    print(f" - {path.name}: title={payload.get('title')!r}, mark={payload.get('mark')!r}, x={x_field!r}, y={y_field!r}, color={color_field!r}")

All DVC/Vega-Lite plot specs generated in this notebook run:
 - /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc_specs/adversarial_vs_benign_accuracy.vl.json
 - /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc_specs/attack_vs_defense_accuracy_heatmap.vl.json
 - /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc_specs/class_labels_apply_fit_vs_accuracy.vl.json
 - /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc_specs/covariance.vl.json
 - /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc_specs/epochs_vs_loss.vl.json
 - /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc_specs/feature_importance.vl.json
 - /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc_specs/hsj_max_iter_vs_accuracy.vl.json
 - /Users/c.meyers/Documents/deckard/docs/notebooks/build/dvc_specs/roc_auc.vl.json

Canonical persist-stage plot targets:
 - roc_auc.vl.json
 - hsj_max_iter_vs_accuracy.vl.json
 - defense_setting_vs_accuracy.vl.json
 - adversarial_vs_b

In [11]:
import json
from pathlib import Path

from IPython.display import IFrame, Markdown, display

from deckard.experiment.dvc import render_dvclive_report

# Use Deckard-native DVCLive report generation and keep rendering in-notebook.
stage_base = generated_path.parent
dvclive_dir = (stage_base / "../outputs/logs/notebook/dvclive").resolve()
result = render_dvclive_report(
    experiment_cfg,
    plugin={
        "enabled": True,
        "dvclive_dir": dvclive_dir.as_posix(),
        "make_summary": True,
        "make_report": True,
        "report_mode": "md",
        "resume": True,
        "save_dvc_exp": False,
        "cache_images": False,
        "monitor_system": False,
    },
)

report_file = result.get("report_file")
if not report_file:
    raise FileNotFoundError("DVCLive report file was not generated")

report_path = Path(report_file)
display(Markdown("### DVCLive Report"))
display(Markdown(f"Generated at: {report_path.as_posix()}"))

if report_path.suffix.lower() == ".html":
    display(IFrame(src=report_path.as_posix(), width="100%", height=900))
elif report_path.suffix.lower() == ".md" and report_path.exists():
    display(Markdown(report_path.read_text()))
elif report_path.suffix.lower() == ".ipynb":
    display(Markdown("Notebook report generated (open the file to inspect):"))
    print(report_path.as_posix())

summary_json = result.get("summary_json")
if summary_json:
    summary_path = Path(summary_json)
    if summary_path.exists():
        display(Markdown("### DVCLive Summary"))
        summary_payload = json.loads(summary_path.read_text())
        print(json.dumps(summary_payload, indent=2)[:2000])

### DVCLive Report

Generated at: /Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/notebook/dvclive/report.md

# DVC Report

metrics.json

|   accuracy |   loss |   step |
|------------|--------|--------|
|       0.81 |   0.19 |      4 |

![static/loss](static/loss.png)

![static/accuracy](static/accuracy.png)


In [10]:
# CLI demonstration (examples/sklearn context) using `!` directives, no Deckard Python API.
# Set RUN_CLI_DEMO=True to execute. Left False by default for fast notebook CI runs.
RUN_CLI_DEMO = True

if RUN_CLI_DEMO:
    !DECKARD_CONFIG_DIR=../../examples/sklearn/config DECKARD_DEFAULT_CONFIG_FILE=default.yaml python -m deckard optimize stage=persist files=default dvc_plugin.enabled=true dvc_plugin.dvclive_dir=outputs/logs/notebook/cli_dvclive dvc_plugin.make_summary=true dvc_plugin.make_report=true dvc_plugin.report_mode=md dvc_plugin.save_dvc_exp=false dvc_plugin.cache_images=false dvc_plugin.monitor_system=false

    cli_dvclive_dir = Path("outputs/logs/notebook/cli_dvclive").resolve()
    print("CLI DVCLive directory:", cli_dvclive_dir.as_posix())
    if cli_dvclive_dir.exists():
        for item in sorted(cli_dvclive_dir.glob("report.*")):
            print(" -", item.as_posix())
else:
    print("Set RUN_CLI_DEMO=True in this cell to execute the CLI DVCLive plugin demo.")

/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
'default.yaml' is validated against ConfigStore schema with the same name.
This behavior is deprecated in Hydra 1.1 and will be removed in Hydra 1.2.
In addition, the automatically matched schema contains a defaults list.
This combination is no longer supported.
See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/automatic_schema_matching for migration instructions.

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
CLI DVCLive directory: /Users/c.meyers/Documents/deckard/docs/notebooks/outputs/logs/notebook/cli_dvclive
